# Consistency between windstorm tracking algorithms – Sensitivity

Production date: 2026-10-21.

Produced by: Enis Gerxhalija & Olivier Burggraaff (National Physical Laboratory).

## 🌍 Use case: C3S Enhanced Windstorm Service (EWS)

## ❓ Quality assessment question
* **How do the tracking algorithms included in the _Windstorm tracks and footprints_ dataset (Hodges, TempestExtremes) compare in terms of sensitivity?**

As part of the Copernicus Climate Change Service's (C3S) Enhanced Windstorm Service (EWS),
the [_Windstorm tracks and footprints derived from reanalysis over Europe between 1940 to present_](https://doi.org/10.24381/bf1f06a9) dataset
is intended to help users in the insurance sector
better understand the risk caused by European windstorms or extra-tropical cyclones (ETCs).
ETCs are large low-pressure systems in mid-latitude regions associated with high winds and heavy rain [[Walker+20](https://doi.org/10.1002/wea.3861)].
These windstorms can cause severe socio-economic damages, with costs in the millions of euros and fatalities ranging from individuals to thousands [[Garnier+18](https://doi.org/10.1016/j.coastaleng.2017.06.014)].

To generate this dataset,
two windstorm tracking algorithms
–
_Hodges_/TRACK [[Hoskins+02](https://doi.org/10.1175/1520-0469(2002)059%3C1041:NPOTNH%3E2.0.CO;2), [Hodges+99](https://doi.org/10.1175/1520-0493(1999)127<1362:ACFFT>2.0.CO;2), [Hodges+95](https://doi.org/10.1175/1520-0493(1995)123<3458:FTOTUS>2.0.CO;2)]
and
_TempestExtremes_ [[Ullrich+21](https://doi.org/10.5194/gmd-14-5023-2021)]
–
were applied to historical meteorological data from ECMWF's fifth-generation reanalysis ERA5 [[Hersbach+20](https://doi.org/10.1002/qj.3803)] to detect windstorm tracks.
The resulting dataset contains three products, namely
the windstorm tracks found by both algorithms,
the associated footprint of each windstorm,
and
yearly summary indicators for administrative regions.

Because the tracking algorithms have different approaches for detecting windstorms,
with
Hodges using 3-hourly 850 hPa relative vorticity
and
TempestExtremes using 6-hourly mean sea level pressure (MSLP),
they are intended to be complementary to each another.
ETCs are highly complex and diverse systems
and
different tracking algorithms have different sensitivities, strengths, and limitations [[Neu+13](https://doi.org/10.1175/BAMS-D-11-00154.1)].
For example,
the MSLP approach generally represents low-frequency, large-scale features,
while vorticity represents high-frequency, small-scale features
[[Neu+13](https://doi.org/10.1175/BAMS-D-11-00154.1), [Hodges+03](https://doi.org/10.1175/1520-0493(2003)131<2012:ACORRD>2.0.CO;2)]

Here,
we compare the outputs of both algorithms in terms of **sensitivity**,
i.e.
how many windstorms they detect (individually and matching)
and the resulting differences in summary indicators.
This comparison is performed as an internal consistency check,
rather than a validation using external data.
The aim is to provide guidance on the choice of algorithm and the consequences of using one over the other, or both at the same time.


## 📢 Quality assessment statement

```{admonition} These are the key outcomes of this assessment
:class: note
* The EWS _Windstorm tracks and footprints_ dataset contains around two thousand windstorms. For 1940–2025, 1 899 windstorms were found using the Hodges algorithm and 871 using TempestExtremes. The Hodges algorithm generally detects more (2.2×) windstorms than TempestExtremes.
* 649 track matchups between algorithms were found, representing 34% of Hodges and 75% of TempestExtremes tracks. The matchup rate is highest in summer and lowest in winter, likely due to the biases in sensitivity of both algorithms. Specific storms may appear in both algorithms (e.g. Ophelia 2017) or only one (e.g. Oscar 2018 and Klaus 2009).
* The difference in sensitivity has a strong temporal component, being largest in winter (DJF, 2.6×) and smallest in summer (JJA, 1.5×). Storm counts from both algorithms tend to follow similar yearly trends but there are some differences. The Hodges algorithm has a stronger seasonal bias (DJF / JJA = 6.2) than TempestExtremes (3.5).
* The difference in sensitivity has a strong spatial component, with TempestExtremes more biased towards the North Atlantic and North Sea, while Hodges detects more storms farther away, such as in eastern and southern Europe.
* The pre-calculated windstorm indicators are consistent over time and space with the track-by-track analysis. For a given region, an internally consistent time series can be derived, allowing for fair comparisons over time with either algorithm. However, the different spatial biases in sensitivity need to be considered when making comparisons across different areas.
```

## 📋 Methodology
This notebook examines the consistency in windstorm characteristics between the two tracking algorithms used in the [_Windstorm tracks and footprints derived from reanalysis over Europe between 1940 to present_](https://doi.org/10.24381/bf1f06a9) dataset.
Here,
we focus on the sensitivity of each algorithm,
by first investigating the individual windstorm tracks
and then the resulting summary indicators.

The analysis and results are organised in the following steps, which are detailed in the sections below:

**[](section-setup)**
* Import all required libraries.
* Define helper functions.
  
**[](section-tracks)**
* Download windstorm tracks from the CDS.
* Compare monthly, seasonal, and yearly windstorm track counts.
* Find and compare track matchups between algorithms.
* Compare geospatial track density.

**[](section-indicators)**
* Download summary indicators from the CDS.
* Compare yearly storm count indicator across NUTS levels.


## 📈 Analysis and results

(section-setup)=
### 1. Code setup
```{note}
This notebook uses [earthkit](https://github.com/ecmwf/earthkit) for
downloading ([earthkit-data](https://github.com/ecmwf/earthkit-data))
and visualising ([earthkit-plots](https://github.com/ecmwf/earthkit-plots)) data.
Because earthkit is in active development, some functionality may change after this notebook is published.
If any part of the code stops functioning, please raise an issue on our GitHub repository so it can be fixed.
```

#### Import required libraries
In this section, we import all the relevant packages needed for running the notebook.

In [1]:
# Input / Output
import earthkit.data as ekd

# General data handling
import calendar
from functools import cache, partial
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr

# Windstorm track matchups
import haversine as hs

# Visualisation
import earthkit.plots as ekp
from earthkit.plots.styles import Style
import matplotlib.pyplot as plt
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["figure.constrained_layout.use"] = True
plt.rcParams["figure.labelweight"] = "bold"
plt.rcParams["figure.titleweight"] = "bold"
plt.rcParams["legend.columnspacing"] = 1.5
plt.rcParams["legend.handletextpad"] = 0.5
from matplotlib.lines import Line2D
from matplotlib.ticker import LogLocator
from cartopy import crs as ccrs
from IPython.display import HTML

# Prevent spurious warnings from Cartopy, Pandas
import warnings
warnings.filterwarnings("ignore", message="Unable to determine extent.*")
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# Visualisation in Jupyter book -- automatically ignored otherwise
try:
    from myst_nb import glue
except ImportError:
    glue = None

# Type hints
from typing import Any, Iterable, Optional
from earthkit.plots.geography.domains import Domain
AnyDomain = (Domain | str)
from pandas.io.formats.style import Styler as PandasStyler

#### Helper functions
This section defines some functions and variables used in the following analysis, allowing code cells in later sections to be shorter and ensuring consistency.

##### Data downloading & (pre-)processing

The following cell contains some pre-defined constants for convenience,
such as a list of variable names in the data:

In [2]:
# Data
TRACKING_ALGORITHMS = ["hodges", "tempest_extremes"]
NUTS_LEVELS         = [0, 1, 2]
EPSG_3035           = 3035  # EPSG projection

# Columns in data
SELECTION_COLUMNS   = ["algorithm", "threshold", "year",]
VARIABLES           = ["storms_number", "mean_wind_gust", "ssi", "normalised_ssi", "area_ratio", "wind_gust_ratio",]
NUTS_VARIABLES      = ["region", "country_code", "NUTS_level", "NUTS_name",]

# Organisation of data
INDEX_COLUMNS       = ["algorithm", "threshold", "region", "year",]
COORDINATES         = ["latitude", "longitude"]

# Seasons
MONTHS              = calendar.month_name[1:]    # Named months of the year; [1:] drops an empty entry at the start
DJF                 = calendar.month_name[12:] + calendar.month_name[1:3]  # Northern Hemisphere winter: December-January-February
MAM                 = calendar.month_name[3:6]   # Northern Hemisphere spring: March-April-May
JJA                 = calendar.month_name[6:9]   # Northern Hemisphere summer: June-July-August
SON                 = calendar.month_name[9:12]  # Northern Hemisphere autumn: September-October-November
SEASONS             = {"DJF": DJF, "MAM": MAM, "JJA": JJA, "SON": SON,}

The following functions download and pre-process NUTS shapefiles from GISCO:

In [3]:
@cache  # Cache results when running notebook multiple times, but lose cache when restarting the kernel
def download_NUTS(*, scale: str="20M", year: str | int="2021", projection: str | int=EPSG_3035,
                  levels: Iterable[int]=NUTS_LEVELS) -> gpd.GeoDataFrame:
    """
    Download NUTS shapefiles from GISCO and store them in a GeoDataFrame.
        scale="20M":       1:20 Million scale
        year="2021":       Year of NUTS; windstorm dataset uses 2021
        projection="3035": EPSG CRS projection code; 3035 is ETRS89-extended / LAEA Europe
        levels=[0, 1, 2]:  NUTS levels to download; windstorm dataset uses 0, 1, 2
    """
    # Set up GISCO URL and request
    base = "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson"
    filename = f"NUTS_RG_{scale}_{year}_{projection}_LEVL_{{level}}.geojson"
    urls = [base + "/" + filename.format(level=level) for level in levels]

    # Download data from GISCO and merge into GeoDataFrame
    nuts_gdf = [gpd.read_file(url) for url in urls]  # List of GeoDataFrames for each NUTS level
    nuts_gdf = gpd.GeoDataFrame(pd.concat(nuts_gdf))
    nuts_gdf = nuts_gdf.rename(columns={"NUTS_ID": "region"})

    return nuts_gdf

The following cell provides some convenience functions for working with Pandas DataFrames,
such as manipulating their indices:

In [4]:
# Helper function: Add dummy index level to existing DataFrame
# (used here to allow certain functions to be re-used with different data)
def add_dummy_index(df: pd.DataFrame, name: str, value: Any) -> pd.DataFrame:
    """ Add a new index level that contains just a dummy value. """
    return pd.concat({value: df}, names=[name])

##### Statistics

The following functions calculate and display statistics on track counts (total, monthly, seasonal, yearly):

In [5]:
# Total number of tracks
def compare_track_counts(data: pd.DataFrame) -> pd.DataFrame:
    """
    Count the number of unique items along the `id` dimension, for each algorithm.
    """
    # Count number of unique track IDs per algorithm
    counts = tracks.groupby(level="algorithm").apply(                                 # Apply separately to each algorithm
                                lambda df: df.index.get_level_values("id").nunique()  # Number of unique values in the "id" index
                           ).to_frame("id")                                           # Convert from Series back to DataFrame

    # Rename columns for clarity and return result
    counts = counts.rename(columns={"id": "storm count"})
    return counts

# Track dates
def summarise_track_dates(data: pd.DataFrame) -> pd.DataFrame:
    """
    For each windstorm track in `data`, get the start/end dates and most important meteorological parameters.
    """
    # Sort by timestamp first
    data = data.sort_values(["algorithm", "id", "time"])

    # Summarise each track (algorithm-id combination)
    summary = tracks.groupby(["algorithm", "id"]).agg(  # Apply aggregation operations to each pair of algorithm-id values
                  time_start=("time", "first"),
                  time_end=("time", "last"),
                  latitude_start=("latitude", "first"),
                  latitude_end=("latitude", "last"),
                  longitude_start=("longitude", "first"),
                  longitude_end=("longitude", "last"),
                  fg10_max=("fg10", "max"),
                  msl_min=("msl", "min"),
              )

    return summary

# Tracks by month / season / year
def compare_track_counts_over_time(summary: pd.DataFrame) -> tuple[pd.DataFrame]:
    """ Compute the monthly / seasonal / yearly windstorm track counts from a table of windstorm track start dates. """
    # Get monthly counts, with names January ... December
    counts_month = (
        summary.groupby(["algorithm", summary["time_start"].dt.month_name(),])  # For each algorithm-month pair
               .size()                                                          # Get the number of elements
               .unstack(fill_value=0)                                           # Move month to column, fill in 0 for empty pairs
    )
    counts_month = counts_month[MONTHS]  # Sort by January ... December using Python's stdlib calendar

    # Get seasonal counts, with names DJF ... SON, from the monthly counts
    counts_season = pd.DataFrame(
        {season: counts_month[months].sum(axis=1) for season, months in SEASONS.items()},
        index=counts_month.index,
    )

    # Get yearly counts, with names 1940 ... 2025
    counts_year = (
        summary.groupby(["algorithm", summary["time_start"].dt.year,])          # For each algorithm-year pair
               .size()                                                          # Get the number of elements
               .unstack(fill_value=0)                                           # Move month to column, fill in 0 for empty pairs
    )

    # Add ratio (Hodges / TempestExtremes)
    for counts in [counts_month, counts_season, counts_year]:
        if "hodges" in counts.index:  # Skip if algorithm names not present, e.g. doing matchups
            counts.loc["ratio"] = counts.loc["hodges"] / counts.loc["tempest_extremes"]

    return counts_month, counts_season, counts_year

def display_track_counts_over_time(counts: pd.DataFrame) -> PandasStyler:
    """ Thin wrapper to display monthly/seasonal/yearly counts more cleanly; counts as int, ratio as float. """
    styled = (
        counts.style
              .format(lambda x: f"{x:.2f}")
              .format(lambda x: f"{x:.0f}", subset=pd.IndexSlice[TRACKING_ALGORITHMS, :])
    )
    return styled

# Within-algorithm: Fraction of winter vs summer storms
def compare_season_fractions(counts_season: pd.DataFrame) -> pd.DataFrame:
    """
    Compute the seasonal fractions (counts in DJF / counts in JJA) for each algorithm.
    """
    # Compute fractions
    counts_season = counts_season.loc[TRACKING_ALGORITHMS]  # Only look at algorithms, not their ratio
    fractions = counts_season["DJF"] / counts_season["JJA"]

    # Convert to DataFrame for nicer formatting in Jupyter Book
    fractions = pd.DataFrame({"DJF / JJA": fractions})
    
    return fractions

The following functions find and analyse track matchups following the method described in [[Hodges+03](https://doi.org/10.1175/1520-0493(2003)131<2012:ACORRD>2.0.CO;2)]:

In [6]:
# Match / Filter by time
def find_overlapping_timestamps(data: pd.DataFrame, *, algorithms: Iterable[str]=TRACKING_ALGORITHMS) -> pd.DataFrame:
    """ Find timestamps that occur in both elements of `algorithms`, e.g. "hodges" and "tempest_extremes". """
    # Split dataset by tracking algorithm
    data_split = [data.loc[alg].reset_index(level="id") for alg in algorithms]

    # Find overlapping timestamps between the two algorithms for each storm ID
    time_overlap = pd.merge(*data_split,  # * splats the individual elements of data_split
                            on="time", how="inner",  # Find all points where the "time" column matches between algorithms
                            suffixes=[f"_{alg}" for alg in algorithms]  # Suffix with _hodges, _tempest_extremes
                           )
    return time_overlap

def match_by_time(data: pd.DataFrame, *, algorithms: Iterable[str]=TRACKING_ALGORITHMS) -> pd.DataFrame:
    """ Identify matching tracks based on temporal overlap, and calculate how much they overlap (% of length). """
    # Count overlapping time points for each track pair
    time_overlap = find_overlapping_timestamps(data, algorithms=algorithms)

    overlap_stats = (
        time_overlap
        .groupby([f"id_{alg}" for alg in algorithms])["time"]  # Loop through all matched pairs of (id_hodges, id_tempest_extremes)
        .agg(  # Calculate aggregated statistics
            length_overlap=pd.Series.nunique,  # Count the number of unique timestamps in each matched pair
            time_start="first",                # Get the first overlapping timestamp
        )
    ).reset_index()  # Turn (id_hodges, id_tempest_extremes) into columns
    
    overlap_stats = overlap_stats.rename(columns={"time": "length_overlap"})  # Rename column for clarity
    overlap_stats.index = overlap_stats.index.rename("id_matchup")            # Rename index for clarity

    # For each matching pair, get the lengths of the corresponding tracks in Hodges and TempestExtremes
    def process_individual_match_by_time(row_overlap: pd.Series) -> pd.Series:
        """ For one match-up row, get the individual track lengths. """
        # Get the original tracks for this ID in both algorithms
        ids_by_algorithm = {alg: row_overlap[f"id_{alg}"] for alg in algorithms}
    
        # Count how many timestamps this track has in both algorithms
        individual_tracks = {alg: data.loc[alg, ID] for alg, ID in ids_by_algorithm.items()}
        individual_lengths = {f"length_{alg}": track["time"].nunique() for alg, track in individual_tracks.items()}
    
        # Bundle into Pandas Series and return
        result = pd.Series(individual_lengths)
        return result

    individual_lengths = overlap_stats.apply(process_individual_match_by_time, axis=1)
    
    # Convert to DataFrame and filter matches
    matches_in_time = pd.concat([overlap_stats, individual_lengths], axis=1)

    # Determine how much overlap there is, for filtering later
    matches_in_time["length_mean"] = matches_in_time[[f"length_{alg}" for alg in algorithms]].mean(axis=1)
    matches_in_time["overlap_fraction"] = matches_in_time["length_overlap"] / matches_in_time["length_mean"]
    
    return matches_in_time

def filter_by_time_overlap(data: pd.DataFrame, *,
                           overlap_col: str="overlap_fraction", threshold: float=0.6) -> pd.DataFrame:
    """ Return only those elements of `data` where `overlap_col` >= threshold. Default is 0.6, from Hodges+03. """
    return data[data[overlap_col] >= threshold]


# Match / Filter by distance
def match_by_mean_distance(data: pd.DataFrame, *, existing_matchups: Optional[pd.DataFrame]=None,
                           algorithms: Iterable[str]=TRACKING_ALGORITHMS) -> pd.DataFrame:
    """
    Identify matching tracks based on temporal overlap, and calculate the mean distance between matched-up timestamps.
    `existing_matchups` can be a DataFrame with known match-ups, e.g. those obtained from matching and filtering on timestamp overlap.
        If provided, only matchups in `existing_matchups` will be considered.
    """
    # Find overlapping timestamps (i.e. timestamps that occur in both algorithms)
    # and calculate the distance between the locations provided by each algorithm
    time_overlap = find_overlapping_timestamps(data, algorithms=algorithms)
    coordinates_by_algorithm = [[f"{coordinate}_{algorithm}"
                                 for coordinate in COORDINATES]
                                for algorithm in algorithms]
    time_overlap_by_algorithm = [time_overlap[coords] for coords in coordinates_by_algorithm]
    time_overlap["mean_distance"] = hs.haversine_vector(*time_overlap_by_algorithm, unit=hs.Unit.KILOMETERS)

    # Calculate the mean distance for each match-up, across all overlapping timestamps
    distances_overlap = time_overlap.groupby([f"id_{alg}" for alg in algorithms])[["mean_distance"]].mean()
    distances_overlap = distances_overlap.reset_index()  # Turn (id_hodges, id_tempest_extremes) into columns
    distances_overlap.index = distances_overlap.index.rename("id_matchup")  # Rename index for clarity

    # If there were existing matchups (e.g. based on time), perform an inner join
    # This will filter out any matchups in `distances_overlap` that are not present in `existing_matchups`
    # (e.g. those filtered out on not having enough overlapping timestamps)
    if existing_matchups is not None:
        distances_overlap = (
            existing_matchups
            .reset_index()  # Move id_matchup to a column
            .merge(distances_overlap,  # Perform the inner join
                   how="inner",  # Find all points where the "time" column matches between algorithms
                   on=[f"id_{alg}" for alg in algorithms],
             )
            .set_index("id_matchup")  # Restore Matchup ID index
        )

    # Select best matches only -- sort by distance (closest first) and pick the first element for every storm
    distances_overlap = distances_overlap.sort_values("mean_distance")
    for alg in algorithms:  # Drop duplicates in Hodges, then in TempestExtremes (if provided in that order)
        distances_overlap = distances_overlap.drop_duplicates(subset=[f"id_{alg}"], keep="first")

    # Return result
    return distances_overlap

def filter_by_mean_distance(data: pd.DataFrame, *,
                            overlap_col: str="mean_distance", threshold: float=300.) -> pd.DataFrame:
    """ Return only those elements of `data` where `overlap_col` < threshold. Default is 300 km, from Hodges+03. """
    return data[data[overlap_col] < threshold]


# Display matchups
def display_track_matchups(data: pd.DataFrame, *, length: int=10) -> pd.DataFrame:
    """
    Display a table of track matchups (as a DataFrame, no HTML styling).
    Removes unwanted columns and applies rounding.
    `length` determines how many elements (at the start + at the end) are shown.
    """
    # Make copy to prevent overwriting data
    display_df = data.copy()  

    # Remove unwanted columns
    display_df = display_df.drop(columns=["length_overlap", "length_mean"])  # Redundant with existing columns

    # Apply number formatting
    display_df["overlap_fraction"] = display_df["overlap_fraction"].map(lambda x: f"{x:.0%}")
    display_df["mean_distance"]    = display_df["mean_distance"].map(   lambda x: f"{x:.0f}")

    return display_df


# Compare matchup counts to overall track counts
def compare_matchup_counts_over_time(*counts_and_matchups: Iterable[pd.DataFrame]) -> tuple[pd.DataFrame]:
    """ For pairs of track (counts, matchups; both DataFrames), calculate the relative number of matchups. """
    relative_matchups = [matchups.loc["matchup"] / counts.loc[TRACKING_ALGORITHMS]
                         for counts, matchups in counts_and_matchups  # Loop over each pair
                        ]
    return relative_matchups

The following functions handle analysis of geospatial track density:

In [7]:
# Pandas DataFrame -> xarray Dataset
def convert_track_counts_to_xarray(data: pd.DataFrame, *, step: float=2.5,
                                   coordinates: Iterable[str]=COORDINATES) -> xr.Dataset:
    """ Convert a DataFrame of storm track counts to xarray, including filling out the grid with 0s. """
    # Use built-in conversion method
    data = data.to_xarray()

    # Fill out grid
    for coord in coordinates:  # Round to 2 digits to prevent floating-point errors
        data[coord] = data[coord].round(2)
    full_coordinates = {coord: np.arange(data[coord].min(), data[coord].max()+0.01, step)  # 2.5° spaced grid in lat, lon
                        for coord in coordinates}
    data = data.reindex(**full_coordinates)

    # Replace NaN values (from where .to_xarray and .reindex added coordinate pairs) with 0
    # since there were 0 storm timestamps there
    data = data.fillna(0)

    return data


def pad_with_nan(data: xr.Dataset, *, step: float=2.5) -> xr.Dataset:
    """ Pad NaN values around an xarray Dataset so the outer edges don't get stretched in pcolormesh plots. """
    # Add padding
    data_padded = data.pad(latitude=(1, 1), longitude=(1, 1), constant_values=np.nan)

    # Adjust coordinates
    lat = data["latitude"].values
    lon = data["longitude"].values
    lat_new, lon_new = [np.concatenate([[coord[0] - step], coord, [coord[-1] + step]]) for coord in (lat, lon)]
    data_padded = data_padded.assign_coords(latitude=lat_new, longitude=lon_new)

    return data_padded


# Spatial aggregation
def binned_track_counts(data: xr.Dataset, *, 
                        lat_min: float=5., lon_min: float=-80., step: float=2.5,
                        id_col: str="id",
                       ) -> xr.Dataset:
    """
    Compute track counts in coarser grid cells.
    By default, start at 5° N and 80° W (lat_min, lon_min) and use a grid size of 2.5° (step).
    """
    # Round latitude, longitude to 2.5°
    # Note this method does not use the COORDINATES constant defined earlier
    tracks_binned = data.copy()
    tracks_binned["latitude"]  = lat_min + np.round((tracks["latitude"]  - lat_min) / step) * step
    tracks_binned["longitude"] = lon_min + np.round((tracks["longitude"] - lon_min) / step) * step

    # Count number of unique tracks per bin
    tracks_binned[id_col] = tracks_binned.index.get_level_values(id_col)  # Extract id index into a column
    track_count_gridded = pd.DataFrame(
        tracks_binned.groupby(["algorithm", "latitude", "longitude",], observed=True)[id_col]
                     .nunique()
                     .rename("track_count")
    )

    # Convert to xarray for easier geospatial analysis
    track_count_gridded = convert_track_counts_to_xarray(track_count_gridded, step=step)

    # Pad with a ring of NaNs for easier plotting (no streaky edges)
    track_count_gridded = pad_with_nan(track_count_gridded, step=step)

    return track_count_gridded

The following functions calculate and display aggregated statistics based on the summary indicators, such as yearly means:

In [8]:
# Calculate difference and ratio between tracking algorithms
def add_difference_and_ratio(data: pd.DataFrame, *,
                             order: Iterable[str]=TRACKING_ALGORITHMS,
                            ) -> pd.DataFrame:
    """ Calculate the difference and ratio between tracking algorithms (order[0] -/ order[1]) and add it to the DataFrame. """
    # Setup: Copy DataFrame format
    first, second = order  # "hodges", "tempest_extremes"
    difference = indicators.loc[first].copy()
    ratio      = indicators.loc[first].copy()

    # Calculate difference, ratio
    variables = [col for col in data.columns if col in VARIABLES]  # Select only those variables that are actually present
    difference[variables] = data.loc[first][variables] - data.loc[second][variables]
    ratio[variables]      = data.loc[first][variables] / data.loc[second][variables]

    # Append to existing DataFrame, convert back to GeoDataFrame
    comparison = pd.concat({"difference": difference, "ratio": ratio}, names=["algorithm"])  # Combine difference and ratio
    data = pd.concat([data, comparison])  # Append to existing data (converts to regular DataFrame)
    data = gpd.GeoDataFrame(data)  # Convert to GeoDataFrame again
    data = data.loc[:, ~data.columns.duplicated()] # Remove duplicate columns

    return data

# Calculate yearly means
def yearly_mean_by_nuts_level(data: pd.DataFrame, indicator: str, *, NUTS_level: int=0,
                              algorithm_order: Iterable[str]=TRACKING_ALGORITHMS+["difference", "ratio"]) -> pd.DataFrame:
    """
    Calculate the yearly mean of `indicator` in a given DataFrame `data`.
    Perform the operation for every entry at one `NUTS_level`, e.g. per country (0).
    Returns a DataFrame with the yearly mean per NUTS region (index) per algorithm (index) per threshold (columns).
    """
    # Select data for this NUTS level
    data_nuts = data[data["NUTS_level"] == NUTS_level]

    # Calculate mean frequency per year
    # The "groupby" call tells Pandas to calculate the mean separately for each combination of algorithm-threshold-region
    yearly_mean = data_nuts.groupby(level=["algorithm", "threshold", "region"]).mean(numeric_only=True)

    # Overwrite ratio with ratio of the means, rather than mean of the ratios
    yearly_mean.loc["ratio"] = (yearly_mean.loc[TRACKING_ALGORITHMS[0]] / yearly_mean.loc[TRACKING_ALGORITHMS[1]]).values
    yearly_mean = yearly_mean.replace([np.inf, -np.inf], np.nan)  # Replace infinities with NaNs

    # Reorganise DataFrame so it's easier to read
    yearly_mean = yearly_mean[[indicator]]  # Select only desired variable
    yearly_mean = yearly_mean.rename(columns={indicator: f"{indicator} (yearly mean)"})  # Rename column to avoid confusion
    yearly_mean = yearly_mean.unstack("threshold")  # Move threshold from index to columns
    yearly_mean = yearly_mean.swaplevel()  # Swap index levels: Region first, then algorithm
    
    # Fix algorithm order to hodges-tempest_extremes-difference for each region
    regions = yearly_mean.index.get_level_values(0).unique()  # Get all regions in DataFrame
    new_index = pd.MultiIndex.from_product(
        [regions, algorithm_order],
        names=["region", "algorithm"]
    )
    yearly_mean = yearly_mean.reindex(new_index)

    return yearly_mean

# Display yearly means in Jupyter Book-friendly style
def display_yearly_mean(yearly_mean: pd.DataFrame) -> HTML:
    """ Display the yearly mean DataFrame with nice styling in a scrollable HTML element. """
    # Setup: Colour by element on the first index (e.g. by country)
    regions = indicators_yearly_mean.index.get_level_values(0).unique()

    region_colours = {region: "#f2f2f2" if i % 2 == 0 else "#ffffff"
                      for i, region in enumerate(regions)}

    def colour_region_row(row):
        colour = region_colours[row.name[0]]
        return [f"background-color: {colour}"] * len(row)

    def colour_region_index(index):
        return [f"background-color: {region_colours[region]}"
                for region in indicators_yearly_mean.index.get_level_values(0)]

    # Apply Pandas style
    styler = (
        yearly_mean.style  # Setup styler object
        .format("{:.2f}")  # Round data to 2 digits
        .format_index(     # Round column labels (threshold) to 1 digit
            lambda x: f"{x:.1f}" if isinstance(x, (int, float)) else x,
            axis=1,
        )
        .apply(colour_region_row, axis=1)  # Colour data according to region
        .apply_index(                      # Colour index according to region
            colour_region_index,
            axis=0,
            level=[0, 1],
        )
    )

    # Format in HTML, return
    html = HTML('<div style="max-height:400px; overflow:auto;">' + styler.to_html() + '</div>')
    return html

# Normalise a DataFrame of indicators by the mean over time / space
def normalise_dataframe(data: pd.DataFrame, columns: Iterable[str]=["storms_number",], *,
                        group_by: Iterable[str]=["algorithm", "threshold",]
                       ) -> pd.DataFrame:
    """
    Normalise one or more columns of a dataframe by the mean over all dimensions except those in group_by.
    Default: Normalise "storms_number" by the mean over all regions and years.
    """
    # Select only the tracking algorithms, and make a copy to avoid overwriting data
    data_relative = data.loc[TRACKING_ALGORITHMS].copy()

    # Calculate the overall mean
    data_columns_mean = (
        data_relative[columns]
                     .groupby(group_by)
                     .mean()
    )

    # Normalise
    data_columns_relative = data_relative[columns] / data_columns_mean

    # Assign to DataFrame
    data_relative[columns] = data_columns_relative

    # Remove columns that have not been normalised
    data_relative = data_relative.drop(columns=[col for col in VARIABLES if col not in columns])

    return data_relative

##### Visualisation

The following cell defines [earthkit-plots styles](https://earthkit-plots.readthedocs.io/en/latest/examples/examples/introduction/05-styles.html) for the windstorm indicators.
These styles define the colour maps and colour bar ranges for each quantity.
It also defines the styles used for the different algorithms in line plots.

In [9]:
# Earthkit styles
_style_storms_number           = {"cmap": plt.cm.cividis, "levels": range(    0, 21, 2),   "extend": "max",}
_style_storms_number_rel       = {"cmap": plt.cm.cividis, "levels": np.arange(0,  3, 0.4), "extend": "max",}

difference_levels     = np.arange(-10.5, 11,    1)
rel_difference_levels = np.arange( -1,    1.01, 0.2)
_style_storms_number_diff      = {"cmap": plt.cm.RdBu,    "levels": difference_levels,     "extend": "both",}
_style_storms_number_rel_diff  = {"cmap": plt.cm.RdBu,    "levels": rel_difference_levels, "extend": "both",}

ratio_levels = np.logspace(np.log10(1/5), np.log10(5), 12)
_style_storms_number_ratio     = {"cmap": plt.cm.RdBu,    "levels": ratio_levels,          "extend": "both",}

track_density_levels = np.logspace(-4, -0.5, 12)
track_density_diff_levels = np.concatenate([-track_density_levels[::-1], track_density_levels])  # -max ... -min ... min ... max
_style_track_count        = {"cmap": plt.cm.cividis, "levels": track_density_levels,       "extend": "both",}
_style_track_count_diff   = {"cmap": plt.cm.RdBu,    "levels": track_density_diff_levels,  "extend": "both",}

styles = {
    "storms_number":                     Style(**_style_storms_number),
    "storms_number_relative":            Style(**_style_storms_number_rel),
    "storms_number_difference":          Style(**_style_storms_number_diff),
    "storms_number_relative_difference": Style(**_style_storms_number_rel_diff),
    "storms_number_ratio":               Style(**_style_storms_number_ratio),
    "storms_number_relative_ratio":      Style(**_style_storms_number_ratio),  # Same as above, on purpose

    "track_count":              Style(**_style_track_count),
    "track_count_difference":   Style(**_style_track_count_diff),
}

# General styles for consistency
# Colours come from Paul Tol's notes (https://sronpersonalpages.nl/~pault/)
ALGORITHM_COLOURS = {"hodges": "#997700", "tempest_extremes": "#6699CC",
                     "difference": "black", "ratio": "black", "matchup": "black",
                    }
ALGORITHM_MARKERS = {"hodges": "s", "tempest_extremes": "^",
                     "difference": "o", "ratio": "o", "matchup": "o",
                    }

The following cell contains some base helper functions (e.g. displaying in Jupyter Notebook or Jupyter Book style):

In [10]:
# Visualisation: Helper functions, general
def _glue_or_show(fig: plt.Figure, glue_label: Optional[str]=None) -> None:
    """
    If `glue` is available, glue the figure using the provided label.
    If not, display the figure in the notebook.
    """
    try:
        glue(glue_label, fig, display=False)
    except TypeError:
        plt.show()
    finally:
        plt.close()

def _add_textbox_to_subplots(text: str, *axs: plt.Axes | ekp.Subplot, right=False) -> None:
    """ Add a text box to each of the specified subplots. """
    # Get the plt.Axes for each ekp.Subplot
    axs = [subplot.ax if isinstance(subplot, ekp.Subplot) else subplot for subplot in axs]

    # Set up location
    x = 0.95 if right else 0.05
    horizontalalignment = "right" if right else "left"

    # Add the text
    for ax in axs:
        ax.text(x, 0.95, text, transform=ax.transAxes,
        horizontalalignment=horizontalalignment, verticalalignment="top",
        bbox={"facecolor": "white", "edgecolor": "black", "boxstyle": "round",
              "alpha": 1})

The following functions are also base helper functions, but specific to geospatial plots.
Administrative boundaries are downloaded from
the _Geographic Information System of the Commission_ (GISCO)
through earthkit-plots,
and are © EuroGeographics.

In [11]:
# Helper constants
DOMAIN_TRACKS = "North Atlantic"

# Visualisation: Helper functions for geospatial plots
def decorate_fig(fig: ekp.Figure, *, title: Optional[str]="") -> None:
    """ Decorate an earthkit figure with land, coastlines, etc. """
    fig.land()
    fig.borders(source="gisco", color="black")
    fig.coastlines(color="black")
    fig.gridlines(linestyle=plt.rcParams["grid.linestyle"])
    fig.title(title)

The following functions display comparisons of windstorm track counts over time:

In [12]:
# Time-based comparison of windstorm track counts
def _plot_counts_over_time(counts_month: pd.DataFrame, counts_season: pd.DataFrame, counts_year: pd.DataFrame, *,
                           algorithms: Iterable[str]=TRACKING_ALGORITHMS,               # Data selection
                           ylabel: str="# Tracks", sharey=False, yscale: str="linear",  # Vertical axes
                           ylim: Optional[Iterable[float]]=None,                        # Vertical axes
                           to_percentage=False,                                         # Convert to % (multiply by 100)
                           glue_label: Optional[str]=None) -> None:
    """
    Plot monthly / seasonal / yearly windstorm track counts / ratios / percentages.
    Don't use this function directly, instead use the partials listed below.
    """
    # Setup: One panel for each time frame (month -> season -> year)
    fig, axs = plt.subplot_mosaic(
        [["Month", "Season"],  # Top row: month, season
         ["Year",  "Year"]],   # Bottom row: year, double width
        gridspec_kw={"hspace": 0.1},  # Padding between rows
        sharey=sharey,  # Share y-axis limits between panels
        figsize=(8, 6),
    )

    axs_sequence = [axs["Month"], axs["Season"], axs["Year"]]

    # Loop over panels to plot data
    for ax, data in zip(axs_sequence, [counts_month, counts_season, counts_year]):
        # Subselect data, apply desired transformations
        data_here = data.loc[algorithms].T  # Only select desired algorithms, transpose so time is on x-axis
        if to_percentage:
            data_here = 100. * data_here

        # Plot algorithms individually
        for algorithm in algorithms:  # Have to loop to allow different markers
            data_here.loc[:, algorithm].plot(ax=ax, 
                                             color=ALGORITHM_COLOURS[algorithm],
                                             marker=ALGORITHM_MARKERS[algorithm], markeredgecolor="black",
                                             legend=False)

        # Decorate panel
        ax.set_yscale(yscale)
        ymin = data_here.min(axis=None)  # Minimum across all values in this DataFrame
        ymax = data_here.max(axis=None)  # Maximum across all values in this DataFrame
        if ylim:  # If ylim supplied by user, use those
            ax.set_ylim(*ylim)
        elif yscale == "linear":    # Fix y-axis to start at 0, end 10% above maximum
            ax.set_ylim(0, ymax*1.1)
        elif yscale == "log":  # Logarithmic scale for fair representation of factors
            lim = 1.1*max(ymax, 1/ymin)  # Expand axes to symmetrically cover full range
            ax.set_ylim(1/lim, lim)

    # Label axes
    for label in ["Month", "Season", "Year"]:  # Add label to x-axis
        axs[label].set_xlabel(label)
    for ax in axs_sequence:  # Add label to y-axis
        ax.set_ylabel(ylabel)
        ax.yaxis.get_label().set_visible(True)
    for label in ["Season",]:  # Season only: move y-labels to right
        axs[label].yaxis.tick_right()  # Move tick labels to the right
        axs[label].tick_params(axis="y", length=0, labelright=True)  # Show tick labels but not the ticks themselves
        axs[label].yaxis.set_label_position("right")  # Force the y-label to be visible even if sharey is True
    for label in ["Month",]:  # Month only: Tilt x-ticks so they don't overlap
        axs[label].tick_params("x", rotation=15)
    fig.align_xlabels()
    fig.align_ylabels()

    # yticks for the log/sharey case (e.g. ratio)
    if sharey and yscale == "log":
        # yticks: From 1 to the limit (up), and the respective fractions (down)
        ymin, ymax = axs["Month"].get_ylim()
        integer_steps = np.arange(1, np.floor(ymax)+0.01, 1, dtype=int)  # All integers from 1 up to and including the maximum
        fractions = 1/integer_steps[1:]  # All fractions down to and including the minimum, but not 1/1 (redundant)
        yticks = [*integer_steps, *fractions]
        yticklabels = [i for i in integer_steps] + [f"1/{i}" for i in integer_steps[1:]]
        # (yticklabels can be done with LaTeX fractions but they don't look great in practice
        ax.set_yticks(yticks, labels=yticklabels)
        ax.minorticks_off()

    # Figure legend (only if showing multiple algorithms)
    if len(algorithms) > 1:
        legend_content = axs["Month"].get_legend_handles_labels()
        fig.legend(*legend_content, loc="center", ncols=2)

    # Show result
    _glue_or_show(fig, glue_label)


# Pre-defined versions for common scenarios
plot_track_counts_over_time = partial(_plot_counts_over_time,
                                      algorithms=TRACKING_ALGORITHMS,
                                      ylabel="# Tracks", sharey=False, yscale="linear", ylim=None,
                                      to_percentage=False,
                                     )

plot_track_count_ratio_over_time = partial(_plot_counts_over_time,
                                           algorithms=["ratio"],
                                           ylabel="ratio", sharey=True, yscale="log", ylim=None,
                                           to_percentage=False,
                                          )

plot_track_matchup_percentage_over_time = partial(_plot_counts_over_time,
                                                  algorithms=TRACKING_ALGORITHMS,
                                                  ylabel="% matched up", sharey=True, yscale="linear", ylim=(0, 105),
                                                  to_percentage=True,
                                                 )

The following functions display geospatial comparisons of individual windstorm tracks:

In [13]:
# Helper functions
def select_track_data_for_ids(track_data: pd.DataFrame, track_ids: dict[str, int]) -> dict[str, pd.DataFrame]:
    """ Get the tracks for each algorithm/ID pair in `track_ids` from the main table `track_data`. """
    # Find data
    tracks_selected = {alg: track_data.loc[alg, ID] for alg, ID in track_ids.items()}

    # Pre-process: Sort by time, reset index, and select variables
    tracks_selected = {alg: track.sort_values("time") for alg, track in tracks_selected.items()}
    tracks_selected = {alg: track.reset_index(drop=True).set_index("time") for alg, track in tracks_selected.items()}
    tracks_selected = {alg: track[["latitude", "longitude"]] for alg, track in tracks_selected.items()}

    return tracks_selected

def plot_track_geospatial(chart: ekp.Map, track: pd.DataFrame, algorithm: str, **kwargs) -> None:
    """
    Plot a windstorm track on a geospatial map, with scatter markers and a dashed line.
    Uses algorithm-specific colours and markers.
    """
    # Setup
    kwargs_shared = {"transform": ccrs.PlateCarree()} | kwargs
    
    # Plot individual points
    chart.ax.scatter(track["longitude"], track["latitude"],
                     color=ALGORITHM_COLOURS[algorithm], marker=ALGORITHM_MARKERS[algorithm],
                     edgecolors="black", s=50,
                     zorder=3,
                     **kwargs_shared)

    # Plot lines connecting points
    chart.ax.plot(track["longitude"], track["latitude"],
                  color=ALGORITHM_COLOURS[algorithm],
                  linestyle="--", linewidth=3,
                  zorder=3-0.1, alpha=0.7,
                  **kwargs_shared)

# Main function
def compare_tracks_geospatial(track_data: pd.DataFrame, track_ids: dict[str, int] | Iterable[dict[str, int]],
                              glue_label: Optional[str]=None) -> None:
    """
    Plot a geospatial comparison between storm tracks in `track_data` from two different tracking algorithms.
    `track_ids` is a Dictionary of track IDs, e.g. {'id_hodges': 1702, 'id_tempest_extremes': 787,}.
    """
    # Check if multiple track_id pairs are being provided or only one
    if isinstance(track_ids, dict):  # Single pair
        track_ids = [track_ids,]  # Create list of length 1, to iterate over

    # Create map
    n_pairs = len(track_ids)
    n_rows = n_pairs // 2 + n_pairs % 2  # One row for every 2 (= n_cols) track_id pairs, plus the remainder (+1 for odd numbers)
    fig = ekp.Figure(rows=n_rows, columns=2, figsize=(10, 1+3.1*n_rows))
    # (figsize is tailored for a 2x2 plot, may not look good for other combinations)

    # Loop over each pair and plot them in their own panel
    for id_pair in track_ids:
        # Remove "id_" prefix
        id_pair = {key.removeprefix("id_"): val for key, val in id_pair.items()}

        # Select desired tracks only
        id_pair = {key: val for key, val in id_pair.items() if val is not None}  # Filter out `None` values (dummies)
        tracks_selected = select_track_data_for_ids(track_data, id_pair)

        # Plot tracks
        chart = fig.add_map(domain=DOMAIN_TRACKS)
        for algorithm, track in tracks_selected.items():
            plot_track_geospatial(chart, track, algorithm)

        # Legend for algorithm lines
        handles = [Line2D([0], [0],
                          color=ALGORITHM_COLOURS[alg], linestyle="--", linewidth=3,
                          marker=ALGORITHM_MARKERS[alg], markersize=8, markerfacecolor=ALGORITHM_COLOURS[alg], markeredgecolor="black",
                          label=alg,)
                   for alg in tracks_selected]
        chart.ax.legend(handles=handles, loc="upper left")

        # Title
        title_ids = " – ".join(f"{alg} ID {ID}" for alg, ID in id_pair.items())
        chart.title(title_ids)

    # Decorate figure
    decorate_fig(fig, title="Storm track comparison")

    # Show result
    _glue_or_show(fig.fig, glue_label)

The following functions display comparisons of spatial windstorm track density:

In [14]:
# Geospatial comparison of track density
def compare_track_density(data: xr.Dataset, variable: str="track_count", *,
                          algorithms: Iterable[str]=TRACKING_ALGORITHMS,
                          glue_label: Optional[str]=None) -> None:
    """
    Plot track densities for both `algorithms` (log scale) and their difference (linear).
    Note: It is possible to make the difference plot appear on a second line,
        but in the current version of earthkit-plots, this causes issues with
        colour bar sizing.
        Loop over enumerate(algorithms) and change the fig.add_map calls to:
            subplot = fig.add_map(0, j)
            subplot = fig.add_map(1, slice(0, 2))
    """
    # Create figure
    fig = ekp.Figure(rows=1, columns=3, figsize=(15, 8))
    
    # Columns: Loop over algorithms
    for alg in algorithms:
        # Select data
        data_alg = data.sel(algorithm=alg)
    
        # Plot data
        subplot = fig.add_map(domain=DOMAIN_TRACKS)
        subplot.grid_cells(data_alg, z=variable, style=styles[variable], ticks=LogLocator(base=10, numticks=5))
    
        # Decorate panel
        subplot.title(alg)
        subplot.legend(location="bottom", label="Track density")
    
    # Last column: Plot difference
    data_diff = data.sel(algorithm=algorithms[0]) - data.sel(algorithm=algorithms[1])
    subplot = fig.add_map(domain=DOMAIN_TRACKS)
    subplot.grid_cells(data_diff, z=variable, style=styles[variable+"_difference"],
                       ticks=np.array([-1e-1, -1e-2, -1e-3, 0, 1e-3, 1e-2, 1e-1]))  # Manual symlog ticks; automatic looks bad
    subplot.title(f"difference\n({algorithms[0]} – {algorithms[1]})")
    subplot.legend(location="bottom", label="Δ Track density")
    
    # Decorate figure
    decorate_fig(fig, title="Algorithm comparison: Storm track timestamp distribution")
    
    # Show result
    _glue_or_show(fig.fig, glue_label)

The following functions display geospatial (choropleth) and time-series comparisons of summary indicators:

In [15]:
# Geospatial comparison
_choropleth_kwargs = {"edgecolor": "black", "linewidth": 0.2,}  # Default kwargs
def compare_indicator_geospatial(data: gpd.GeoDataFrame, indicator: str, *,
                                 threshold: float=0.0, year: int=2024,
                                 comparisons: Iterable[str]=["difference", "ratio",],
                                 relative: bool=False,
                                 domain: AnyDomain="Europe",
                                 glue_label: Optional[str]=None) -> None:
    """
    Plot data for one variable `indicator` from a GeoDataFrame `data`.
    Resulting figure will have:
        rows: NUTS_LEVELS (0, 1, 2 by default)
        cols: TRACKING_ALGORITHMS (hodges, tempest_extremes, difference by default)
    Figure is a snapshot for one `year` and one `threshold`.
    """
    # Select data for one threshold and year only
    data = data.loc[:, threshold, :, year]

    # Setup: Style, choose absolute or relative
    indicator_style = indicator+"_relative" if relative else indicator

    # Setup: 3x4 Figure
    nalgs, ncomps = len(TRACKING_ALGORITHMS), len(comparisons)
    nrows = nalgs + ncomps
    ncols = len(NUTS_LEVELS)
    fig = ekp.Figure(rows=nrows, columns=ncols, figsize=(1+3*ncols, 1+3.5*nrows))
    subplots = np.array([[fig.add_map(domain=domain) for i in range(ncols)] for j in range(nrows)])

    # Columns: Loop over NUTS levels
    for level, ax_col in zip(NUTS_LEVELS, subplots.T):
        # Select data
        data_nuts = data[data["NUTS_level"] == level]

        # Rows 0, 1: Loop over algorithms
        for j, algorithm in enumerate(TRACKING_ALGORITHMS):
            subplot = ax_col[j]
            subplot.choropleth(data_nuts.loc[algorithm], z=indicator, style=styles[indicator_style], **_choropleth_kwargs,)

        # Rows 2, 3: Plot difference, ratio
        for j, comparison in enumerate(comparisons, start=nalgs):
            subplot = ax_col[j]
            subplot.choropleth(data_nuts.loc[comparison], z=indicator,
                               style=styles[f"{indicator_style}_{comparison}"],
                               **_choropleth_kwargs)

    # Labels in all panels
    for label, ax_row in zip(TRACKING_ALGORITHMS+comparisons, subplots):
        _add_textbox_to_subplots(label, *ax_row, right=True)

    # Colour bar on the right
    for subplot in subplots[:nalgs, -1]:
        subplot.legend(location="right", label=indicator)
    for subplot, comparison in zip(subplots[nalgs:, -1], comparisons):
        subplot.legend(location="right", label=comparison)

    # Decorate figure
    decorate_fig(fig, title=f"Algorithm comparison for {indicator} indicator (year: {year}, threshold: {threshold} m/s)")

    # Show result
    _glue_or_show(fig.fig, glue_label)


# Time-series comparison
def compare_indicator_timeseries(data: gpd.GeoDataFrame, indicator: str, region: str, *,
                                 glue_label: Optional[str]=None) -> None:
    """
    Plot data for one variable `indicator` from a GeoDataFrame `data`.
    Resulting figure will have 4 (2x2) panels, one for each threshold.
    Figure is made for a single NUTS `region`, e.g. "IE", "DEA", "AL01".
    """
    # Setup: 2x2 figure for thresholds
    fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(8, 8), gridspec_kw={"hspace": 0.1},
                            sharex=True, sharey=True,)

    # Loop over panels, plot one threshold in each
    for ax, threshold in zip(axs.ravel(), indicators.index.levels[1]):  # Second level index has all thresholds
        # Select data for this threshold and region only
        data_nuts = data.loc[:, threshold, region, :][indicator]

        # Plot algorithms
        for algorithm in TRACKING_ALGORITHMS:
            data_nuts.loc[algorithm].plot(ax=ax, color=ALGORITHM_COLOURS[algorithm], markeredgecolor="black",
                                          marker=ALGORITHM_MARKERS[algorithm], markersize=6,
                                          label=algorithm,)

        # Panel title
        ax.set_title(f"Threshold: {threshold} m/s")

    # Label x-axis on bottom row
    for ax in axs[-1]:
        ax.set_xlabel("Year")

    # Label y-axis on left-hand column
    for ax in axs[:, 0]:
        ax.set_ylabel(indicator)

    # Start y-axis at 0; automatically transferred to all panels because sharey=True
    axs[0, 0].set_ylim(ymin=0)

    # Figure title
    region_full_name = data.loc[:, :, region]["NUTS_name"].iloc[0]  # Retrieve region name from GeoDataFrame
    fig.suptitle(f"Algorithm comparison for {indicator} indicator (region: {region}; {region_full_name})")

    # Figure legend
    fig.legend(*axs[0, 0].get_legend_handles_labels(), loc="center")
    
    # Show result
    _glue_or_show(fig, glue_label)

(section-tracks)=
### 2. Windstorm tracks
We first compare the windstorm tracks found by the Hodges and TempestExtremes algorithms.
The following cell defines the years for which we will download data;
by default, this is 1940–2025, but these years can be modified below when running the notebook yourself:

In [16]:
# Customisation
# Modify the range below to change the years considered
year_start, year_end = 1940, 2025

#### Download windstorm tracks
We download the windstorm tracks from the
[sis-european-wind-storm-reanalysis](https://doi.org/10.24381/bf1f06a9)
CDS entry.
This is a simple table which can be loaded into a Pandas [DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html).
For each algorithm (Hodges, TempestExtremes),
this table contains all timestamps with corresponding location (latitude, longitude) and some fields describing physical characteristics at that timestamp and location (see the [Product User Guide](https://confluence.ecmwf.int/x/jHaXGw#WindstormtracksandfootprintsderivedfromreanalysisoverEuropebetween1940topresent:ProductUserGuide-Windstormtrack) for details).

Note that the `id` column in this table is defined separately for each algorithm.
For example, the windstorm with `id` 42 in the Hodges list is likely _not_ the same as the windstorm with that `id` in the TempestExtremes list.

This notebook uses [earthkit-data](https://github.com/ecmwf/earthkit-data) to download files from the CDS.
If you intend to run this notebook multiple times, it is highly recommended that you [enable caching](https://earthkit-data.readthedocs.io/en/latest/concepts/caching.html) to prevent having to download the same files multiple times.
If you prefer not to use earthkit, the following requests can also be used with the [cdsapi module](https://cds.climate.copernicus.eu/how-to-api).
In either case (earthkit-data or cdsapi), it is required to set up a CDS account and API key as explained [on the CDS website](https://cds.climate.copernicus.eu/how-to-api).

In [17]:
# Define request for variables, based on CDS request form
dataset = "sis-european-wind-storm-reanalysis"

request = {
    "product": "windstorm_track",
    "variable": "all",
    "tracking_algorithm": ["hodges", "tempestextremes"],
    "event_aggregation": "single_event",
    "year":  [f"{year}"     for year  in range(year_start, year_end+1)],
    "month": [f"{month:02}" for month in range(1, 13)],
    "day":   [f"{day:02}"   for day   in range(1, 32)],
}

# Download data and load into Pandas
tracks = ekd.from_source("cds", dataset, request)
tracks = tracks.to_pandas()

# Pre-process: Use Pandas timestamps and set a custom index
tracks["time"] = pd.to_datetime(tracks["time"])
tracks = tracks.set_index(["algorithm", "id"])  # Reindex

# Display in notebook
tracks

time  latitude  longitude       fg10  \
algorithm        id                                                        
hodges           1   1940-02-22 00:00:00     63.25     -24.00  10.452867   
                 1   1940-02-22 06:00:00     63.50     -25.00  12.599429   
                 1   1940-02-22 12:00:00     62.75     -11.25   5.919555   
                 1   1940-02-22 18:00:00     63.00     -12.00   7.741296   
                 1   1940-02-23 00:00:00     62.75     -12.25   4.190840   
...                                  ...       ...        ...        ...   
tempest_extremes 871 2025-10-29 00:00:00     57.50      -7.50   7.256220   
                 871 2025-10-29 06:00:00     57.75      -6.50   5.611463   
                 871 2025-10-29 12:00:00     58.50      -2.25   7.861186   
                 871 2025-10-30 12:00:00     55.75      15.75   7.850687   
                 871 2025-10-30 18:00:00     56.50      19.00   5.625481   

                           lsm        msl  
algorithm        id                        
hodges           1    0.000000  98962.875  
                 1    0.000000  98905.190  
                 1    0.000000  98809.125  
                 1    0.000000  98755.560  
                 1    0.000000  98655.310  
...                        ...        ...  
tempest_extremes 871  0.203919  98960.750  
                 871  0.058715  98859.560  
                 871  0.000000  98899.120  
                 871  0.000000  98927.310  
                 871  0.000000  98920.690  

[34254 rows x 6 columns]

#### Windstorm count comparison
First, we compare the overall number of storms detected by each algorithm:

In [18]:
compare_track_counts(tracks)

,storm count
algorithm,
hodges,1899
tempest_extremes,871


It is immediately clear from this comparison that the Hodges algorithm detects many more windstorms overall than TempestExtremes,
with 2.2× the storm count.

Next, we compare storm counts over monthly, seasonal, and yearly time scales.
For this, each individual storm is assigned a single timestamp based on when it was first detected.

In [19]:
# Generate summary table
track_summary = summarise_track_dates(tracks)

# Display in notebook
track_summary

time_start            time_end  latitude_start  \
algorithm        id                                                            
hodges           1   1940-02-22 00:00:00 1940-02-24 12:00:00           63.25   
                 2   1940-03-12 00:00:00 1940-03-15 06:00:00           47.75   
                 3   1940-03-14 00:00:00 1940-03-15 18:00:00           65.25   
                 4   1940-03-17 12:00:00 1940-03-20 12:00:00           48.25   
                 5   1940-04-06 00:00:00 1940-04-08 12:00:00           59.25   
...                                  ...                 ...             ...   
tempest_extremes 867 2025-06-26 12:00:00 2025-06-29 06:00:00           52.75   
                 868 2025-08-07 18:00:00 2025-08-09 12:00:00           62.00   
                 869 2025-10-03 00:00:00 2025-10-05 18:00:00           49.75   
                 870 2025-10-18 06:00:00 2025-10-20 18:00:00           48.75   
                 871 2025-10-27 12:00:00 2025-10-30 18:00:00           58.75   

                      latitude_end  longitude_start  longitude_end   fg10_max  \
algorithm        id                                                             
hodges           1           58.75           -24.00          17.50  12.599429   
                 2           53.00           -16.75          21.75  16.327724   
                 3           60.00            -5.00           0.00  18.169280   
                 4           56.75           -29.25          -4.00  28.481495   
                 5           67.00           -26.25           8.75  23.800018   
...                            ...              ...            ...        ...   
tempest_extremes 867         67.00           -41.75           3.50   7.519233   
                 868         62.25           -24.75          -4.50  16.972857   
                 869         57.25           -24.75          11.00  25.967460   
                 870         51.75           -23.25          -0.75  19.189919   
                 871         56.50           -22.75          19.00  13.957808   

                        msl_min  
algorithm        id              
hodges           1    98551.810  
                 2    97628.190  
                 3    98380.625  
                 4    96242.690  
                 5    97699.250  
...                         ...  
tempest_extremes 867  97936.120  
                 868  98190.000  
                 869  94543.310  
                 870  97429.690  
                 871  98723.440  

[2770 rows x 8 columns]

Both algorithms detect considerably more windstorms in the northern autumn (SON) and winter (DJF) months than in the other half of the year
(Figure {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-count>`),
which is a known bias in some tracking algorithms [[Neu+13](https://doi.org/10.1175/BAMS-D-11-00154.1)].
The year-to-year windstorm counts from both algorithms generally follow each other,
with peaks and troughs in the same locations,
albeit with some notable exceptions (e.g. 1990, 2010).
This behaviour is reflected in the ratio of track counts
(Figure {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-ratio>`),
which is generally around 2 but fluctuates as high as 5 and as low as 1 in the 1940–2025 range.
Seasonally, the ratio varies from 1.45 in summer (JJA) to 2.59 in winter:

In [20]:
# Compute monthly / seasonal / yearly track counts
track_counts_month, track_counts_season, track_counts_year = compare_track_counts_over_time(track_summary)

# Plot monthly / seasonal / yearly track counts
plot_track_counts_over_time(track_counts_month, track_counts_season, track_counts_year,
                            glue_label=f"indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-count",
                           )

# Plot monthly / seasonal / yearly track count ratio
plot_track_count_ratio_over_time(track_counts_month, track_counts_season, track_counts_year,
                                 glue_label=f"indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-ratio",
                                )

# Display seasonal track counts table
display_track_counts_over_time(track_counts_season)

,DJF,MAM,JJA,SON
algorithm,,,,
hodges,849,327,136,587
tempest_extremes,328,159,94,290
ratio,2.59,2.06,1.45,2.02


The bias towards winter windstorms,
defined here as the ratio of the DJF storm count over the JJA one,
is almost twice as large in Hodges as in TempestExtremes:

In [21]:
compare_season_fractions(track_counts_season)

,DJF / JJA
algorithm,
hodges,6.242647
tempest_extremes,3.489362


This bias is caused by physical differences between winter and summer ETCs,
such as
winter ETCs generally being more intense and thus easier to identify and track for some algorithms,
and
summer having more shallow ETCs that are excluded by some algorithms
[[Walker+20](https://doi.org/10.1002/wea.3861), [Neu+13](https://doi.org/10.1175/BAMS-D-11-00154.1)].

In summary,
the Hodges algorithm is generally more sensitive than TempestExtremes,
detecting more windstorms both overall and on different timescales.
However,
the algorithms react differently to different windstorm characteristics,
causing different biases in their sensitivity across the seasons (e.g. winter vs. summer).

::::{tab-set}
:::{tab-item} Track counts
:sync: counts
```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-count
:figwidth: 700px
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-count"

Number of tracks found by the Hodges and TempestExtremes algorithms
over the period 1940–2025
by calendar month, season, and year.
```
:::
:::{tab-item} Track count ratio
:sync: ratio
```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-ratio
:figwidth: 700px
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-ratio"

Ratio of tracks found by the Hodges and TempestExtremes algorithms
(Hodges / TempestExtremes)
over the period 1940–2025
by calendar month, season, and year.
```
:::
::::

#### Track-to-track matchup comparison
Next,
we compare the behaviour of the two algorithms for windstorm tracks found by both.
This requires applying a track-to-track matchup method,
for which we use the one described in [[Hodges+03](https://doi.org/10.1175/1520-0493(2003)131<2012:ACORRD>2.0.CO;2)],
which works as follows:
* For each pair of tracks (Hodges–TempestExtremes), find the number of matching timestamps.
* Keep only those pairs with ≥60% (relative to the mean track length between the two algorithms) overlap.
* For each remaining pair of tracks, compute the mean distances between matched-up timestamps.
* Keep only those pairs where the mean distance is <300 km.


In [22]:
# Find matchups in time and keep only those that overlap >= 60%
# To change this threshold, change the value of `threshold=0.6` to your desired value
track_matchups_time = match_by_time(tracks)
track_matchups_time = filter_by_time_overlap(track_matchups_time, threshold=0.6)

# Calculate mean distances between matchups and keep only those with mean distances < 300 km
# To change this threshold, change the value of `threshold=300` to your desired value
track_matchups_distance = match_by_mean_distance(tracks, existing_matchups=track_matchups_time)
track_matchups_distance = filter_by_mean_distance(track_matchups_distance, threshold=300)

# Display result
display_track_matchups(track_matchups_distance)

,id_hodges,id_tempest_extremes,time_start,length_hodges,length_tempest_extremes,overlap_fraction,mean_distance
id_matchup,,,,,,,
1115,1702,787,2017-10-15 06:00:00,9,9,100%,0
1087,1666,770,2016-01-01 00:00:00,6,8,86%,0
576,887,426,1982-11-08 18:00:00,12,12,100%,0
383,602,291,1970-05-20 00:00:00,8,15,61%,0
380,598,287,1970-02-11 18:00:00,8,8,100%,0
...,...,...,...,...,...,...,...
331,508,250,1965-12-30 06:00:00,18,25,74%,274
455,708,344,1975-03-01 06:00:00,14,20,76%,277
630,955,462,1985-04-11 12:00:00,11,19,67%,280


In [23]:
# Calculate total number of matchups
n_matchup = track_matchups_distance.index.nunique()

# Loop over tracking algorithms, display % of matches
for alg in TRACKING_ALGORITHMS:
    n_alg = tracks.loc[alg].index.nunique()  # Number of unique IDs for this index
    fraction = n_matchup / n_alg  # How many of this algorithm's tracks show up in the match-up table

    # Display result
    padding = " "*(16 - len(alg))  # Spacing in print statement to make it look nicer
    print(f"{alg}: {padding} {n_matchup:} / {n_alg:>4} tracks ({fraction:.1%})")

hodges:            649 / 1899 tracks (34.2%)
tempest_extremes:  649 /  871 tracks (74.5%)


Using this method,
649 track matchups are found.
This number may vary when using a different algorithm or different threshold parameters.
These matchups represent 34.2% of all tracks found by Hodges and 74.5% of those found by TempestExtremes.
As before, the number of matchups and the matchup fraction vary over time
(Figures {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-withmatchup>`, {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-matchup>`),
being essentially a combination of the trends for the individual algorithms.

Overall,
agreement appears to be best in summer,
which may be because both algorithms detect the fewest storms in summer
–
any detected summer ETCs are more likely to be relatively ‘easy’ detections,
and thus more likely to appear in _both_ or _neither_ algorithm than in one but not the other.

Matchup pairs fall into several categories
(Figure {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-geospatial-comparison>`).
The most general case (top left; Hodges 508, TempestExtremes 250) has both algorithms first detecting the windstorm at different times and locations,
e.g. because it is not yet clearly defined or because there are multiple systems close to each other,
and then converging on a common path.
These are cases where the overlap fraction is between 60–100% and the mean distance is non-zero.
For some windstorms (top right; Hodges 1702, TempestExtremes 787; [Storm Ophelia](https://en.wikipedia.org/wiki/Hurricane_Ophelia_(2017))),
the algorithms match exactly in time and space.
Lastly, there are examples where storms are detected by only Hodges (bottom left; Hodges 1525; [Storm Klaus](https://en.wikipedia.org/wiki/Cyclone_Klaus)) or TempestExtremes (bottom right; TempestExtremes 796; [Storm Oscar](https://en.wikipedia.org/wiki/2018_Atlantic_hurricane_season#Hurricane_Oscar)).

Using only matched-up pairs is a good solution when confidence is more important than coverage,
for example when investigating windstorm dynamics or hazards.
Coverage may be more important in other use cases,
such as estimating total storm counts or when one is particularly interested in storms that are difficult to detect with the biases present in these algorithms.
In the latter case,
one can use the full catalogue provided in this dataset or choose only one algorithm.

In [24]:
# Get starting months
track_matchups_distance["month_start"] = track_matchups_distance["time_start"].dt.month_name()

# Assign dummy "algorithm" index so that compare_track_counts_over_time can be re-used
track_matchups_distance = add_dummy_index(track_matchups_distance, "algorithm", "matchup")

# Compute monthly / seasonal / yearly matchup counts
matchups_month, matchups_season, matchups_year = compare_track_counts_over_time(track_matchups_distance)

# Display absolute matchup counts
combined_month, combined_season, combined_year = (  # Temporarily combine the matchup counts with the individual algorithm counts
    pd.concat([track_counts_month,  matchups_month]),
    pd.concat([track_counts_season, matchups_season]),
    pd.concat([track_counts_year,   matchups_year]),
)
plot_track_counts_over_time(combined_month, combined_season, combined_year,
                            algorithms=TRACKING_ALGORITHMS+["matchup",],  # Plot algorithm track counts + matchup count
                            glue_label="indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-withmatchup",
                           )

# Compute relative matchup counts (matchup / algorithm)
matchups_relative_month, matchups_relative_season, matchups_relative_year = (
    compare_matchup_counts_over_time([track_counts_month, matchups_month],
                                     [track_counts_season, matchups_season],
                                     [track_counts_year, matchups_year],
                                    )
)

# Display relative matchup counts
plot_track_matchup_percentage_over_time(matchups_relative_month, matchups_relative_season, matchups_relative_year,
                                        glue_label="indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-matchup",
                                       )

In [25]:
# Specify the storm tracks you want to compare using the respective Hodges and TempestExtremes IDs as below.
# These can be derived by manual inspection of the comparison table.
storm_example   = {"id_hodges":  508, "id_tempest_extremes":  250,}  # From matchup table, December 1965
storm_ophelia   = {"id_hodges": 1702, "id_tempest_extremes":  787,}  # Storm Ophelia,      October  2017
storm_klaus     = {"id_hodges": 1525, "id_tempest_extremes": None,}  # Storm Klaus,        January  2009
storm_oscar     = {"id_hodges": None, "id_tempest_extremes":  796,}  # Storm Oscar,        November 2018

storms_to_plot = [storm_example, storm_ophelia, storm_klaus, storm_oscar]

# Plot
compare_tracks_geospatial(tracks, storms_to_plot,
                          glue_label="indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-geospatial-comparison",
                         )

::::{tab-set}
:::{tab-item} Track counts
:sync: counts
```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-withmatchup
:figwidth: 700px
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-withmatchup"

Number of tracks found by the Hodges and TempestExtremes algorithms,
as well as the number of matched-up tracks,
over the period 1940–2025
by calendar month, season, and year.
```
:::
:::{tab-item} Matchup fraction
:sync: ratio
```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-matchup
:figwidth: 700px
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-matchup"

Fraction of storm tracks from each algorithm
(Hodges, TempestExtremes)
that have a matchup with the other
over the period 1940–2025
by calendar month, season, and year.
Fractions >100% may occur due to mismatching timestamps,
e.g. a storm first detected on 31 December 2015 by TempestExtremes and on 1 January 2016 by Hodges.
```
:::
:::{tab-item} Individual track comparisons
:sync: tracks
```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-geospatial-comparison
:figwidth: 700px
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-geospatial-comparison"

Geospatial comparison of storm tracks from each algorithm
(Hodges, TempestExtremes)
for individual windstorm events.
Includes storms found in both algorithms (top row)
and found in only one algorithm (bottom).
```
:::
::::

#### Spatial track density
Lastly,
we compare the spatial density of windstorm tracks across the domain of the dataset (North Atlantic) between the two algorithms.
For each grid cell,
this density is computed as the number of unique windstorm tracks that pass through that cell,
normalised by the total number of tracks [[Pérez-Alarcón+24](https://doi.org/10.1016/j.envsoft.2024.106027), [Neu+13](https://doi.org/10.1175/BAMS-D-11-00154.1)],
both calculated individually for each algorithm.
For visual clarity,
this calculation is performed on a coarsened grid (2.5° × 2.5°).

Both algorithms show a visually similar distribution
(Figure {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-density>`).
The main difference is that TempestExtremes has relatively more density over the seas just west of mainland Europe (Iceland–Spain) while Hodges has more over northern and eastern Europe (Norway–Belarus).
This is likely the result of the Hodges algorithm tracking storms for longer
(as discussed in
a forthcoming quality assessment for this dataset),
thus capturing the tail ends of storms over these areas.

In [26]:
# Compute track counts on 2.5° grid
track_count_gridded = binned_track_counts(tracks, step=2.5)

# Total number of storm tracks for each algorithm
total_track_count = (
    tracks.index.to_frame(index=False)
          .groupby("algorithm")["id"]
          .nunique()
          .to_xarray()  # For compatibility with track_count_gridded
)

# Density of storm track timestamps for each algorithm, relative to total
track_density_gridded = track_count_gridded / total_track_count

# Plot
compare_track_density(track_density_gridded,
                      glue_label="indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-density")

```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-density
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-density"

Geospatial density of windstorm track timestamps
found by the Hodges and TempestExtremes algorithms,
aggregated to a 2.5° grid,
and the difference (Hodges – TempestExtremes).
A negative difference (red) means TempestExtremes has a higher density,
positive (blue) means Hodges has a higher density.
The density is relative to the total number of windstorm tracks in each algorithm individually.
```

(section-indicators)=
### 3. Summary indicators
In addition to the individual windstorm tracks and footprints,
the [sis-european-wind-storm-reanalysis](https://doi.org/10.24381/bf1f06a9)
CDS entry
provides pre-calculated yearly windstorm indicators describing storm number and intensity.
These are
organised by NUTS
(Nomenclature of Territorial Units for Statistics)
level 0, 1, and 2
region.
Of these indicators,
`storms_number`
counts the number of storms that have affected the given region.
A storm event is considered to affect a region if its decontaminated footprint intersects the region.

Indicators are provided for
both tracking algorithms
(Hodges and TempestExtremes)
and
for four different thresholds
(0.0, 15.6, 20.0, and 25.0 m/s),
i.e. considering only those parts of a storm where the wind gust exceeds this threshold.
The thresholds allow the user to choose if they want to consider all storms or only particularly intense ones, for example.


#### Download windstorm summary indicators
We download the full windstorm summary indicator table from the
CDS.
Because this is a simple table,
the data are loaded into a Pandas [DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html).


In [27]:
# Define request for variables, based on CDS request form
dataset = "sis-european-wind-storm-reanalysis"

request = {
    "product": "windstorm_summary_indicator",
    "variable": "all",
    "tracking_algorithm": ["hodges", "tempestextremes"],
}

# Download data and load into Pandas
indicators = ekd.from_source("cds", dataset, request)
indicators = indicators.to_pandas()

# Display in notebook
indicators

,year,region,threshold,storms_number,mean_wind_gust,ssi,normalised_ssi,area_ratio,wind_gust_ratio,algorithm
0,1940,FR,0.0,10,19.450012,3.954249e+09,0.292018,0.980225,0.667874,hodges
1,1940,FR,15.6,10,21.074094,5.029804e+09,0.371447,0.980225,0.723641,hodges
2,1940,FR,20.0,9,23.179559,6.692985e+09,0.494271,0.980225,0.795939,hodges
3,1940,FR,25.0,5,26.403130,9.891692e+09,0.730493,0.980225,0.906630,hodges
4,1940,HR,0.0,2,13.676930,1.374967e+08,0.243874,0.959770,0.633382,hodges
...,...,...,...,...,...,...,...,...,...,...
341243,2025,NO07,25.0,2,28.244418,2.161430e+09,1.543523,1.000000,1.155680,tempest_extremes
341244,2025,NO0B,0.0,0,NaN,NaN,NaN,NaN,NaN,tempest_extremes
341245,2025,NO0B,15.6,0,NaN,NaN,NaN,NaN,NaN,tempest_extremes
341246,2025,NO0B,20.0,0,NaN,NaN,NaN,NaN,NaN,tempest_extremes


Visualising the NUTS regions requires the corresponding shapefiles,
which can be retrieved from [GISCO](https://ec.europa.eu/eurostat/web/nuts).
These ancillary data are loaded into a GeoPandas [GeoDataFrame](https://geopandas.org/en/latest/docs/reference/api/geopandas.GeoDataFrame.html)
and combined with the indicators table:

In [28]:
# Download NUTS shapefiles
nuts_gdf = download_NUTS()

# Merge NUTS polygons into data
indicators = nuts_gdf.merge(indicators, on="region")

# Rename some columns for ease of use
indicators = indicators.rename(columns={"LEVL_CODE": "NUTS_level", "CNTR_CODE": "country_code", "NAME_LATN": "NUTS_name"})

# Filter only desired columns, in pre-set order
indicators = indicators[[*SELECTION_COLUMNS, *NUTS_VARIABLES, *VARIABLES, "geometry",]]

# Re-index data for easier selection
indicators = indicators.set_index(INDEX_COLUMNS)

# Display in notebook
indicators

country_code  NUTS_level  \
algorithm        threshold region year                            
hodges           0.0       FR     1940           FR           0   
                 15.6      FR     1940           FR           0   
                 20.0      FR     1940           FR           0   
                 25.0      FR     1940           FR           0   
                 0.0       FR     1941           FR           0   
...                                             ...         ...   
tempest_extremes 25.0      NO0B   2024           NO           2   
                 0.0       NO0B   2025           NO           2   
                 15.6      NO0B   2025           NO           2   
                 20.0      NO0B   2025           NO           2   
                 25.0      NO0B   2025           NO           2   

                                                     NUTS_name  storms_number  \
algorithm        threshold region year                                          
hodges           0.0       FR     1940                  France             10   
                 15.6      FR     1940                  France             10   
                 20.0      FR     1940                  France              9   
                 25.0      FR     1940                  France              5   
                 0.0       FR     1941                  France              7   
...                                                        ...            ...   
tempest_extremes 25.0      NO0B   2024  Jan Mayen and Svalbard              0   
                 0.0       NO0B   2025  Jan Mayen and Svalbard              0   
                 15.6      NO0B   2025  Jan Mayen and Svalbard              0   
                 20.0      NO0B   2025  Jan Mayen and Svalbard              0   
                 25.0      NO0B   2025  Jan Mayen and Svalbard              0   

                                        mean_wind_gust           ssi  \
algorithm        threshold region year                                 
hodges           0.0       FR     1940       19.450012  3.954249e+09   
                 15.6      FR     1940       21.074094  5.029804e+09   
                 20.0      FR     1940       23.179559  6.692985e+09   
                 25.0      FR     1940       26.403130  9.891692e+09   
                 0.0       FR     1941       19.011019  3.708104e+09   
...                                                ...           ...   
tempest_extremes 25.0      NO0B   2024             NaN           NaN   
                 0.0       NO0B   2025             NaN           NaN   
                 15.6      NO0B   2025             NaN           NaN   
                 20.0      NO0B   2025             NaN           NaN   
                 25.0      NO0B   2025             NaN           NaN   

                                        normalised_ssi  area_ratio  \
algorithm        threshold region year                               
hodges           0.0       FR     1940        0.292018    0.980225   
                 15.6      FR     1940        0.371447    0.980225   
                 20.0      FR     1940        0.494271    0.980225   
                 25.0      FR     1940        0.730493    0.980225   
                 0.0       FR     1941        0.273840    0.984368   
...                                                ...         ...   
tempest_extremes 25.0      NO0B   2024             NaN         NaN   
                 0.0       NO0B   2025             NaN         NaN   
                 15.6      NO0B   2025             NaN         NaN   
                 20.0      NO0B   2025             NaN         NaN   
                 25.0      NO0B   2025             NaN         NaN   

                                        wind_gust_ratio  \
algorithm        threshold region year                    
hodges           0.0       FR     1940         0.667874   
                 15.6      FR     1940         0.723641   
                 20.0      FR

#### Yearly storm count comparison
We calculate the per-point difference and ratio between algorithms
(Hodges – TempestExtremes, Hodges / TempestExtremes)
and append them to the GeoDataFrame as additional entries along the `algorithm` index:

In [29]:
# Calculate difference and ratio, add to original GeoDataFrame
indicators = add_difference_and_ratio(indicators)

# Display result
indicators

country_code  NUTS_level  \
algorithm threshold region year                            
hodges    0.0       FR     1940           FR           0   
          15.6      FR     1940           FR           0   
          20.0      FR     1940           FR           0   
          25.0      FR     1940           FR           0   
          0.0       FR     1941           FR           0   
...                                      ...         ...   
ratio     25.0      NO0B   2024           NO           2   
          0.0       NO0B   2025           NO           2   
          15.6      NO0B   2025           NO           2   
          20.0      NO0B   2025           NO           2   
          25.0      NO0B   2025           NO           2   

                                              NUTS_name  storms_number  \
algorithm threshold region year                                          
hodges    0.0       FR     1940                  France           10.0   
          15.6      FR     1940                  France           10.0   
          20.0      FR     1940                  France            9.0   
          25.0      FR     1940                  France            5.0   
          0.0       FR     1941                  France            7.0   
...                                                 ...            ...   
ratio     25.0      NO0B   2024  Jan Mayen and Svalbard            NaN   
          0.0       NO0B   2025  Jan Mayen and Svalbard            NaN   
          15.6      NO0B   2025  Jan Mayen and Svalbard            NaN   
          20.0      NO0B   2025  Jan Mayen and Svalbard            NaN   
          25.0      NO0B   2025  Jan Mayen and Svalbard            NaN   

                                 mean_wind_gust           ssi  normalised_ssi  \
algorithm threshold region year                                                 
hodges    0.0       FR     1940       19.450012  3.954249e+09        0.292018   
          15.6      FR     1940       21.074094  5.029804e+09        0.371447   
          20.0      FR     1940       23.179559  6.692985e+09        0.494271   
          25.0      FR     1940       26.403130  9.891692e+09        0.730493   
          0.0       FR     1941       19.011019  3.708104e+09        0.273840   
...                                         ...           ...             ...   
ratio     25.0      NO0B   2024             NaN           NaN             NaN   
          0.0       NO0B   2025             NaN           NaN             NaN   
          15.6      NO0B   2025             NaN           NaN             NaN   
          20.0      NO0B   2025             NaN           NaN             NaN   
          25.0      NO0B   2025             NaN           NaN             NaN   

                                 area_ratio  wind_gust_ratio  \
algorithm threshold region year                                
hodges    0.0       FR     1940    0.980225         0.667874   
          15.6      FR     1940    0.980225         0.723641   
          20.0      FR     1940    0.980225         0.795939   
          25.0      FR     1940    0.980225         0.906630   
          0.0       FR     1941    0.984368         0.652800   
...                                     ...              ...   
ratio     25.0      NO0B   2024         NaN              NaN   
          0.0       NO0B   2025         NaN              NaN   
          15.6      NO0B   2025         NaN              NaN   
          20.0      NO0B   2025         NaN              NaN   
          25.0      NO0B   2025         NaN              NaN   

                                                                          geometry  
algorithm threshold region year                                                     
hodges    0.0       FR     1940  MULTIPOLYGON (((3767290.578 3128769.785, 37818...  
          15.6      FR     1940  MULTIPOLYGON (((3767290.578 3128769.785, 37818...  
          20.0      FR     1940  MULTIPOLYGON (((3767290.578 

Next, we calculate the yearly mean number of storms affecting each region.
The following table only shows NUTS level 0 (country) for brevity,
but the code cell can be modified to NUTS level 1 or 2 if running the notebook yourself.

In [30]:
# Choose which NUTS level (0, 1, or 2) to run the analysis for
# although note that the table may get excessively long at 2
NUTS_level = 0

# Calculate yearly mean storms_number
indicators_yearly_mean = yearly_mean_by_nuts_level(indicators, "storms_number", NUTS_level=NUTS_level)

# Display yearly mean storms_number
display_yearly_mean(indicators_yearly_mean)

As in the previous analysis,
the Hodges algorithm generally detects 2–3× as many storms as TempestExtremes in most areas and for all four thresholds.
The Hodges / TempestExtremes storm count ratio is closest to 1 around the North Sea
(Figures {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-geospatial-0.0>`, {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-geospatial-20.0>`),
which is the area most affected by ETCs
(Figure {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-density>`).
This is likely a combination of a real bias in sensitivity
–
both algorithms being equally sensitive to (the types of) storms occurring in the North Sea
but
Hodges being more sensitive to (the types or stages of) storms in areas farther away
–
and noise, with there being fewer storms in those outlying areas and thus noisier statistics.
All windstorm tracking algorithms have their own unique spatial biases,
e.g. some detecting fewer storms in the Mediterranean due to some threshold criteria in the algorithm tending to exclude the types of storms that occur there [[Pérez-Alarcón+24](https://doi.org/10.1016/j.envsoft.2024.106027)].

The relative number of windstorms
(Figure {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-geospatial-0.0-rel>`;
relative to the mean over all years and NUTS regions for a given threshold and algorithm)
is highest in Northwestern Europe for both algorithms,
with TempestExtremes showing the strongest concentration bias (red areas in the difference and ratio panels).
These biases must be accounted for
to make fair comparisons across Europe.
This is a fundamental difficulty in the field of windstorm tracking
[[Pérez-Alarcón+24](https://doi.org/10.1016/j.envsoft.2024.106027), [Walker+20](https://doi.org/10.1002/wea.3861), [Neu+13](https://doi.org/10.1175/BAMS-D-11-00154.1)].

For the example NUTS 0 region of Éire/Ireland,
the time series
(Figure {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-timeseries>`)
is similar to the overall track count time series seen before
(Figure {numref}`{number} <indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-track-count-withmatchup>`).
This correspondence changes when evaluating different regions,
particularly ones farther from the North Atlantic / North Sea area because the number of windstorms in outlying areas will be less tightly correlated to the total.
This can be customised in the next code cell when running this notebook yourself.

In conclusion,
the pre-calculated windstorm indicators provide a convenient summary of windstorm events
that is consistent over time and space with the track-by-track analysis.
For a given region,
an internally consistent time series can be derived,
allowing for fair comparisons over time with either algorithm.
However,
the algorithms have different spatial biases in their sensitivity
that need to be considered when making comparisons across different areas of Europe.

In [31]:
# Calculate relative distribution of storms_number
indicators_relative = normalise_dataframe(indicators)
indicators_relative = add_difference_and_ratio(indicators_relative)  # Re-calculate difference and ratio between algorithms

# Geospatial maps of storms_number by NUTS region for 2 thresholds (can be customised below)
for threshold in [0.0, 20.0,]:
    # Setup: Glue label (for display in Jupyter Book)
    glue_label_absolute = f"indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-geospatial-{threshold:.1f}"
    glue_label_relative = f"indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-geospatial-{threshold:.1f}-rel"

    # Plot absolute, then relative
    compare_indicator_geospatial(indicators, "storms_number",
                                 threshold=threshold, year=2024,
                                 glue_label=glue_label_absolute)

    compare_indicator_geospatial(indicators_relative, "storms_number",
                                 threshold=threshold, year=2024,
                                 relative=True,
                                 glue_label=glue_label_relative)

In [32]:
# The `region` argument controls which geographic area is plotted.
# To change the region, simply modify "IE" to any other region, such as:
#   region="UK"   - United Kingdom (NUTS 0)
#   region="DEA"  - Nordrhein-Westfalen, Germany (NUTS 1)
#   region="AL01" - Veri, Albania (NUTS 2)

compare_indicator_timeseries(indicators, "storms_number",
                             region="IE",
                             glue_label="indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-timeseries")

::::{tab-set}
:::{tab-item} Geospatial (> 0.0 m/s)
:sync: 0.0
```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-geospatial-0.0
:figwidth: 700px
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-geospatial-0.0"

Comparison between the Hodges (top) and TempestExtremes (second from top) algorithms,
including the difference Hodges – TempestExtremes (second from bottom) and ratio Hodges / TempestExtremes (bottom),
for the `storms_number` summary indicator
across Europe.
Includes all storms (threshold of 0.0 m/s).
Columns correspond to NUTS levels 0–2 (left to right).
```
:::
:::{tab-item} Geospatial (> 20.0 m/s)
:sync: 20.0
```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-geospatial-20.0
:figwidth: 700px
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-geospatial-20.0"

Comparison between the Hodges (top) and TempestExtremes (second from top) algorithms,
including the difference Hodges – TempestExtremes (second from bottom) and ratio Hodges / TempestExtremes (bottom),
for the `storms_number` summary indicator
across Europe.
Includes only storms that pass the 20.0 m/s threshold.
Columns correspond to NUTS levels 0–2 (left to right).
```
:::
:::{tab-item} Geospatial (relative)
:sync: relative
```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-geospatial-0.0-rel
:figwidth: 700px
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-geospatial-0.0-rel"

Comparison between the Hodges (top) and TempestExtremes (second from top) algorithms,
including the difference Hodges – TempestExtremes (second from bottom) and ratio Hodges / TempestExtremes (bottom),
for the `storms_number` summary indicator
across Europe.
Includes all storms (threshold of 0.0 m/s).
Columns correspond to NUTS levels 0–2 (left to right).
```
:::
:::{tab-item} Time series
:sync: counts
```{glue:figure} indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-timeseries
:figwidth: 700px
:name: "indicator_sis-european-wind-storm-reanalysis_consistency_q01_fig-summary-indicator-timeseries"

Comparison between the Hodges (brown) and TempestExtremes (blue) algorithms
for the `storms_number` summary indicator
over time.
Includes only storms that affected the "IE" NUTS 0 region (Éire/Ireland).
```
:::
::::

## ℹ️ If you want to know more

### Key resources
The CDS catalogue entries for the data used were:
* Windstorm tracks and footprints derived from reanalysis over Europe between 1940 to present: [sis-european-wind-storm-reanalysis](https://doi.org/10.24381/bf1f06a9)

More about windstorm tracking algorithms:
* [IMILAST: A Community Effort to Intercompare Extratropical Cyclone Detection and Tracking Algorithms](https://doi.org/10.1175/BAMS-D-11-00154.1)
* [The numerous approaches to tracking extratropical cyclones and the challenges they present](https://doi.org/10.1002/wea.3861)
* Hodges / TRACK
  * [Feature Tracking on the Unit Sphere](https://doi.org/10.1175/1520-0493(1995)123<3458:FTOTUS>2.0.CO;2)
  * [Adaptive Constraints for Feature Tracking](https://doi.org/10.1175/1520-0493(1999)127<1362:ACFFT>2.0.CO;2)
  * [New Perspectives on the Northern Hemisphere Winter Storm Tracks](https://doi.org/10.1175/1520-0469(2002)059<1041:NPOTNH>2.0.CO;2)
* TempestExtremes
  * [TempestExtremes v2.1: a community framework for feature detection, tracking, and analysis in large datasets](https://doi.org/10.5194/gmd-14-5023-2021)

Code libraries used:
* [GeoPandas](https://geopandas.org/en/stable/)
* [Haversine](https://github.com/mapado/haversine)
* [earthkit](https://earthkit.ecmwf.int/)
  * [earthkit-data](https://github.com/ecmwf/earthkit-data)
  * [earthkit-plots](https://github.com/ecmwf/earthkit-plots)

### References
[[Garnier+18](https://doi.org/10.1016/j.coastaleng.2017.06.014)] E. Garnier, P. Ciavola, T. Spencer, O. Ferreira, C. Armaroli, and A. McIvor, ‘Historical analysis of storm events: Case studies in France, England, Portugal and Italy’, Coastal Engineering, vol. 134, pp. 10–23, Apr. 2018, doi: 10.1016/j.coastaleng.2017.06.014.

[[Hersbach+20](https://doi.org/10.1002/qj.3803)] H. Hersbach et al., ‘The ERA5 global reanalysis’, Quarterly Journal of the Royal Meteorological Society, vol. 146, no. 730, pp. 1999–2049, May 2020, doi: 10.1002/qj.3803.

[[Hodges+95](https://doi.org/10.1175/1520-0493(1995)123<3458:FTOTUS>2.0.CO;2)] K. I. Hodges, ‘Feature Tracking on the Unit Sphere’, Monthly Weather Review, vol. 123, no. 12, pp. 3458–3465, Dec. 1995, doi: 10.1175/1520-0493(1995)123<3458:FTOTUS>2.0.CO;2.

[[Hodges+99](https://doi.org/10.1175/1520-0493(1999)127<1362:ACFFT>2.0.CO;2)] K. I. Hodges, ‘Adaptive Constraints for Feature Tracking’, Monthly Weather Review, vol. 127, no. 6, pp. 1362–1373, Jun. 1999, doi: 10.1175/1520-0493(1999)127<1362:ACFFT>2.0.CO;2.

[[Hodges+03](https://doi.org/10.1175/1520-0493(2003)131<2012:ACORRD>2.0.CO;2)] K. I. Hodges, B. J. Hoskins, J. Boyle, and C. Thorncroft, ‘A Comparison of Recent Reanalysis Datasets Using Objective Feature Tracking: Storm Tracks and Tropical Easterly Waves’, Monthly Weather Review, vol. 131, no. 9, pp. 2012–2037, Sep. 2003, doi: 10.1175/1520-0493(2003)131<2012:ACORRD>2.0.CO;2.

[[Hoskins+02](https://doi.org/10.1175/1520-0469(2002)059<1041:NPOTNH>2.0.CO;2)] B. J. Hoskins and K. I. Hodges, ‘New Perspectives on the Northern Hemisphere Winter Storm Tracks’, Journal of the Atmospheric Sciences, vol. 59, no. 6, pp. 1041–1061, Mar. 2002, doi: 10.1175/1520-0469(2002)059<1041:NPOTNH>2.0.CO;2.

[[Neu+13](https://doi.org/10.1175/BAMS-D-11-00154.1)] U. Neu et al., ‘IMILAST: A Community Effort to Intercompare Extratropical Cyclone Detection and Tracking Algorithms’, Bulletin of the American Meteorological Society, vol. 94, no. 4, pp. 529–547, Apr. 2013, doi: 10.1175/BAMS-D-11-00154.1.

[[Pérez-Alarcón+24](https://doi.org/10.1016/j.envsoft.2024.106027)] A. Pérez-Alarcón, P. Coll-Hidalgo, R. M. Trigo, R. Nieto, and L. Gimeno, ‘CyTRACK: An open-source and user-friendly python toolbox for detecting and tracking cyclones’, Environmental Modelling & Software, vol. 176, p. 106027, May 2024, doi: 10.1016/j.envsoft.2024.106027.

[[Ullrich+21](https://doi.org/10.5194/gmd-14-5023-2021)] P. A. Ullrich, C. M. Zarzycki, E. E. McClenny, M. C. Pinheiro, A. M. Stansfield, and K. A. Reed, ‘TempestExtremes v2.1: a community framework for feature detection, tracking, and analysis in large datasets’, Geoscientific Model Development, vol. 14, no. 8, pp. 5023–5048, Aug. 2021, doi: 10.5194/gmd-14-5023-2021.

[[Walker+20](https://doi.org/10.1002/wea.3861)] E. Walker, D. Mitchell, and W. Seviour, ‘The numerous approaches to tracking extratropical cyclones and the challenges they present’, Weather, vol. 75, no. 11, pp. 336–341, Nov. 2020, doi: 10.1002/wea.3861.
